# Task 3.1 — Prepare a Classification Target

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('clean_dataset.csv')

# Target: is_absent (already binary-scaled -> convert back to clean 0/1 labels)
y = (df['is_absent'] > 0).astype(int)   # 1 = absent, 0 = present

# Drop target + status_ABSENT/status_PRESENT
# (these are literally a one-hot encoding of is_absent -> data leakage!)
X = df.drop(columns=['is_absent', 'status_ABSENT', 'status_PRESENT'])

print('Class balance:')
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y keeps the same 90/10 class ratio in both train and test

Class balance:
is_absent
0    0.902778
1    0.097222
Name: proportion, dtype: float64


# Task 3.2 — Logistic Regression (Baseline)

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print('Logistic Regression Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred):.3f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Logistic Regression Results:
  Accuracy:  1.000
  Precision: 1.000
  Recall:    1.000
  F1 Score:  1.000

Confusion Matrix:
[[65  0]
 [ 0  7]]


# Task 3.3 — Tree-Based Classifiers

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)
print(f'Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.3f}')

rf_clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(f'Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')

# Random Forest usually more stable than a single tree (less overfitting)

Decision Tree Accuracy: 1.000
Random Forest Accuracy: 1.000


# Task 3.4 — K-Nearest Neighbors (KNN)

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# KNN is distance-based -> MUST scale features first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # use TRAIN scaler, don't refit

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
print(f'KNN Accuracy: {accuracy_score(y_test, y_pred_knn):.3f}')

# Test multiple values of k
for k in [3, 5, 7, 9, 11]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_scaled))
    print(f'  k={k}: accuracy={acc:.3f}')

KNN Accuracy: 0.931
  k=3: accuracy=0.958
  k=5: accuracy=0.931
  k=7: accuracy=0.917
  k=9: accuracy=0.903
  k=11: accuracy=0.903


# Task 3.5 — Compare All Classification Models

In [5]:
clf_results = {
    'Logistic Regression': accuracy_score(y_test, y_pred),
    'Decision Tree': accuracy_score(y_test, y_pred_dt),
    'Random Forest': accuracy_score(y_test, y_pred_rf),
    'KNN': accuracy_score(y_test, y_pred_knn),
}

clf_results_df = pd.DataFrame(clf_results.items(), columns=['Model', 'Accuracy'])
clf_results_df = clf_results_df.sort_values('Accuracy', ascending=False)
print(clf_results_df)

clf_results_df.to_csv('classification_comparison.csv', index=False)

                 Model  Accuracy
0  Logistic Regression  1.000000
1        Decision Tree  1.000000
2        Random Forest  1.000000
3                  KNN  0.930556
